# ÉPICA 1: Gestión de Documentos Duplicados

Herramienta de anonimización documental para ECOOO - Gestión de duplicados

## Objetivo
Identificar, registrar y organizar documentos duplicados para optimizar el proceso de anonimización

## Importes y configuración

In [1]:
import os
import json
import hashlib
import shutil
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, asdict
import pandas as pd

# Configuración
DOCS_DIR = Path("/Users/usuario/Desktop/anonimizacion_ecooo/datos")
DUPLICATES_DIR = Path("/Users/usuario/Desktop/anonimizacion_ecooo/duplicados")
OUTPUT_DIR = Path("/Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output")
DUPLICATES_DIR.mkdir(exist_ok=True)

print(f"Directorio de documentos: {DOCS_DIR}")
print(f"Directorio de duplicados: {DUPLICATES_DIR}")
print(f"Directorio de salida: {OUTPUT_DIR}")

Directorio de documentos: /Users/usuario/Desktop/anonimizacion_ecooo/datos
Directorio de duplicados: /Users/usuario/Desktop/anonimizacion_ecooo/duplicados
Directorio de salida: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output


## Arquitectura de Datos - Gestión de Duplicados

In [2]:
@dataclass
class DuplicateGroup:
    """Grupo de archivos duplicados"""
    hash_value: str
    files: list
    total_size: int
    redundancy_size: int
    primary_file: str = None
    
    def __post_init__(self):
        if not self.primary_file and self.files:
            self.primary_file = self.files[0]

@dataclass
class DuplicateRegistry:
    """Registro completo de duplicados"""
    total_groups: int
    total_duplicate_files: int
    total_original_files: int
    total_redundancy_bytes: int
    duplicate_groups: list

print("✅ Estructuras de datos definidas")

✅ Estructuras de datos definidas


## Identificación de Duplicados

In [3]:
class DuplicateDetector:
    """Detecta y gestiona archivos duplicados"""
    
    def __init__(self, docs_dir: Path):
        self.docs_dir = docs_dir
        self.file_hashes = defaultdict(list)
        self.file_sizes = {}
    
    def scan_for_hashes(self) -> None:
        """Escanea directorio y calcula hashes"""
        print("Calculando hashes de archivos...")
        
        file_count = 0
        for root, dirs, files in os.walk(self.docs_dir):
            for file in files:
                if file.startswith('.') or file.endswith('.zip'):
                    continue
                
                filepath = Path(root) / file
                try:
                    with open(filepath, 'rb') as f:
                        file_hash = hashlib.md5(f.read()).hexdigest()
                    
                    rel_path = str(filepath.relative_to(self.docs_dir))
                    self.file_hashes[file_hash].append(rel_path)
                    self.file_sizes[rel_path] = filepath.stat().st_size
                    
                    file_count += 1
                    if file_count % 500 == 0:
                        print(f"  Procesados: {file_count} archivos")
                except:
                    pass
        
        print(f"✅ Hashes calculados: {file_count} archivos")
    
    def get_duplicates(self) -> dict:
        """Retorna solo los duplicados"""
        duplicates = {h: files for h, files in self.file_hashes.items() if len(files) > 1}
        return duplicates
    
    def analyze_duplicates(self, duplicates: dict) -> dict:
        """Analiza estadísticas de duplicados"""
        total_duplicate_files = sum(len(files) for files in duplicates.values())
        
        total_redundancy = 0
        for hash_val, files in duplicates.items():
            file_size = self.file_sizes.get(files[0], 0)
            # Redundancia = tamaño archivo * (cantidad - 1)
            total_redundancy += file_size * (len(files) - 1)
        
        return {
            "total_groups": len(duplicates),
            "total_duplicate_files": total_duplicate_files,
            "total_redundancy_bytes": total_redundancy,
            "total_redundancy_mb": total_redundancy / (1024*1024)
        }

# Ejecutar detección
detector = DuplicateDetector(DOCS_DIR)
detector.scan_for_hashes()

duplicates = detector.get_duplicates()
analysis = detector.analyze_duplicates(duplicates)

print(f"\n📊 ANÁLISIS DE DUPLICADOS:")
print(f"  Grupos de duplicados: {analysis['total_groups']}")
print(f"  Archivos duplicados: {analysis['total_duplicate_files']}")
print(f"  Redundancia: {analysis['total_redundancy_mb']:.2f} MB")

Calculando hashes de archivos...


  Procesados: 500 archivos


  Procesados: 1000 archivos


  Procesados: 1500 archivos


  Procesados: 2000 archivos


  Procesados: 2500 archivos


  Procesados: 3000 archivos


  Procesados: 3500 archivos
✅ Hashes calculados: 3668 archivos

📊 ANÁLISIS DE DUPLICADOS:
  Grupos de duplicados: 655
  Archivos duplicados: 1570
  Redundancia: 615.66 MB


## Copiar Duplicados a Carpeta Destinada

In [4]:
class DuplicateOrganizer:
    """Organiza y copia archivos duplicados"""
    
    def __init__(self, source_dir: Path, dest_dir: Path):
        self.source_dir = source_dir
        self.dest_dir = dest_dir
        self.dest_dir.mkdir(exist_ok=True)
    
    def copy_duplicates(self, duplicates: dict) -> dict:
        """Copia archivos duplicados a carpeta destino"""
        print(f"Copiando duplicados a {self.dest_dir}...")
        
        copied = 0
        for hash_val, files in duplicates.items():
            # Crear subcarpeta con hash
            dup_folder = self.dest_dir / hash_val[:8]
            dup_folder.mkdir(exist_ok=True)
            
            for rel_path in files:
                source_file = self.source_dir / rel_path
                dest_file = dup_folder / Path(rel_path).name
                
                try:
                    shutil.copy2(source_file, dest_file)
                    copied += 1
                except Exception as e:
                    print(f"  ⚠️ Error copiando {rel_path}: {e}")
        
        print(f"✅ {copied} archivos copiados")
        return {"files_copied": copied}

# Ejecutar copia
organizer = DuplicateOrganizer(DOCS_DIR, DUPLICATES_DIR)
copy_result = organizer.copy_duplicates(duplicates)

print(f"\n📂 Estructura de duplicados:")
print(f"  Raíz: {DUPLICATES_DIR}")
print(f"  Carpetas: duplicados/[hash_8_chars]/")
print(f"  Archivos copiados: {copy_result['files_copied']}")

Copiando duplicados a /Users/usuario/Desktop/anonimizacion_ecooo/duplicados...


✅ 1570 archivos copiados

📂 Estructura de duplicados:
  Raíz: /Users/usuario/Desktop/anonimizacion_ecooo/duplicados
  Carpetas: duplicados/[hash_8_chars]/
  Archivos copiados: 1570


## Generar Registro de Duplicados

In [5]:
class DuplicateRegistry:
    """Genera registros de duplicados"""
    
    def __init__(self, duplicates: dict, file_sizes: dict):
        self.duplicates = duplicates
        self.file_sizes = file_sizes
    
    def create_registry(self) -> dict:
        """Crea registro completo"""
        total_files = sum(len(files) for files in self.duplicates.values())
        total_originals = len(self.duplicates)  # Una copia es el original
        total_redundancy = 0
        
        # Top 20 duplicados (ordenados por redundancia)
        sorted_dups = []
        for hash_val, files in self.duplicates.items():
            file_size = self.file_sizes.get(files[0], 0)
            redundancy = file_size * (len(files) - 1)
            total_redundancy += redundancy
            
            sorted_dups.append({
                "hash": hash_val[:8],
                "copies": len(files),
                "size_per_file_kb": file_size / 1024,
                "total_redundancy_kb": redundancy / 1024,
                "files": files
            })
        
        sorted_dups.sort(key=lambda x: -x['total_redundancy_kb'])
        
        registry = {
            "summary": {
                "total_duplicate_groups": len(self.duplicates),
                "total_duplicate_files": total_files,
                "total_original_files": total_originals,
                "total_redundancy_mb": total_redundancy / (1024*1024),
                "storage_saved_if_deduplicated_mb": total_redundancy / (1024*1024)
            },
            "top_20_redundant": sorted_dups[:20]
        }
        
        return registry
    
    def export_as_csv(self, output_path: Path) -> None:
        """Exporta duplicados como CSV"""
        rows = []
        for hash_val, files in self.duplicates.items():
            file_size = self.file_sizes.get(files[0], 0)
            for i, filepath in enumerate(files):
                rows.append({
                    "hash_group": hash_val[:8],
                    "file_number": i + 1,
                    "total_copies": len(files),
                    "filepath": filepath,
                    "size_kb": file_size / 1024
                })
        
        df = pd.DataFrame(rows)
        df.to_csv(output_path, index=False)
        print(f"✅ CSV exportado: {output_path}")

# Ejecutar registro
registry_gen = DuplicateRegistry(duplicates, detector.file_sizes)
registry = registry_gen.create_registry()

print(f"\n📋 REGISTRO DE DUPLICADOS:")
print(json.dumps(registry['summary'], indent=2))

# Exportar JSON
json_path = OUTPUT_DIR / "registro_duplicados.json"
with open(json_path, 'w') as f:
    json.dump(registry, f, indent=2)
print(f"\n✅ Registro JSON: {json_path}")

# Exportar CSV
csv_path = OUTPUT_DIR / "duplicados_detallado.csv"
registry_gen.export_as_csv(csv_path)


📋 REGISTRO DE DUPLICADOS:
{
  "total_duplicate_groups": 655,
  "total_duplicate_files": 1570,
  "total_original_files": 655,
  "total_redundancy_mb": 615.6610250473022,
  "storage_saved_if_deduplicated_mb": 615.6610250473022
}

✅ Registro JSON: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output/registro_duplicados.json
✅ CSV exportado: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output/duplicados_detallado.csv


## Estrategia de Manejo de Duplicados

In [6]:
strategy = """
ESTRATEGIA DE MANEJO DE DUPLICADOS
====================================

1. IDENTIFICACIÓN
   - Se identificaron 655 grupos de duplicados
   - 1570 archivos duplicados
   - Redundancia total: {:.2f} MB

2. ORGANIZACIÓN
   - Carpeta: duplicados/
   - Estructura: duplicados/[hash_8_chars]/[archivos]
   - Permite análisis y auditoría de duplicados

3. ANONIMIZACIÓN
   OPCIÓN A: Procesar una sola vez
   - Anonimizar el archivo "primario"
   - Copiar resultado a todas las ubicaciones originales
   - Ventaja: Optimiza tiempo de procesamiento
   - Ventaja: Garantiza consistencia

   OPCIÓN B: Eliminar duplicados antes
   - Eliminar duplicados de datos/
   - Anonimizar solo archivos únicos
   - Ventaja: Reduce tamaño total
   - Ventaja: Acelera procesamiento masivo

4. AUDITORÍA
   - Archivos: 
     * registro_duplicados.json (registro completo)
     * duplicados_detallado.csv (análisis detallado)
   - Carpeta: duplicados/ (copias para verificación)
""".format(analysis['total_redundancy_mb'])

print(strategy)

# Exportar estrategia
strategy_path = OUTPUT_DIR / "estrategia_duplicados.txt"
with open(strategy_path, 'w') as f:
    f.write(strategy)
print(f"\n✅ Estrategia exportada: {strategy_path}")


ESTRATEGIA DE MANEJO DE DUPLICADOS

1. IDENTIFICACIÓN
   - Se identificaron 655 grupos de duplicados
   - 1570 archivos duplicados
   - Redundancia total: 615.66 MB

2. ORGANIZACIÓN
   - Carpeta: duplicados/
   - Estructura: duplicados/[hash_8_chars]/[archivos]
   - Permite análisis y auditoría de duplicados

3. ANONIMIZACIÓN
   OPCIÓN A: Procesar una sola vez
   - Anonimizar el archivo "primario"
   - Copiar resultado a todas las ubicaciones originales
   - Ventaja: Optimiza tiempo de procesamiento
   - Ventaja: Garantiza consistencia

   OPCIÓN B: Eliminar duplicados antes
   - Eliminar duplicados de datos/
   - Anonimizar solo archivos únicos
   - Ventaja: Reduce tamaño total
   - Ventaja: Acelera procesamiento masivo

4. AUDITORÍA
   - Archivos: 
     * registro_duplicados.json (registro completo)
     * duplicados_detallado.csv (análisis detallado)
   - Carpeta: duplicados/ (copias para verificación)


✅ Estrategia exportada: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/o